# Watch PySAD raise alerts on NYC taxi demand

[![Open in JupyterLite](https://jupyterlite.rtfd.io/en/latest/_static/badge.svg)](https://pyodide.github.io/jupyterlite/lab/index.html?fromURL=https%3A//raw.githubusercontent.com/selimfirat/pysad/master/examples/nyc_taxi_alerts.ipynb)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/selimfirat/pysad/blob/master/examples/nyc_taxi_alerts.ipynb)

This notebook runs [PySAD](https://github.com/selimfirat/pysad) in your browser. [Pyodide](https://pyodide.org) runs Python, NumPy, SciPy and scikit-learn in the page, so you don't need to install anything. The data comes with the page.

The series counts the passengers of New York City taxis in each 30 minutes, from July 2014 to January 2015. It comes from the [Numenta Anomaly Benchmark (NAB)](https://github.com/numenta/NAB), which marks five anomaly windows: the NYC marathon, Thanksgiving, Christmas, New Year's Day and a snowstorm.

To start, select **Run > Run All Cells**. The first run loads Python and takes a few seconds.

In [ ]:
%pip install pysad

## Load the series

`Data` reads the series and labels the values in NAB's anomaly windows with 1. `download=False` makes it read the copy that comes with the page.

In [ ]:
import numpy as np

from pysad.utils import Data

X, y = Data("data").get_data("nab/realKnownCause/nyc_taxi.csv", download=False)
times = np.loadtxt(
    "data/nab/realKnownCause/nyc_taxi.csv",
    delimiter=",",
    skiprows=1,
    usecols=0,
    dtype="datetime64[m]",
)
print(
    f"{len(X)} values from {times[0]} to {times[-1]}, {int(y.sum())} of them in NAB's anomaly windows"
)

## Stream the series and raise alerts

`RelativeEntropy` compares the histogram of the last 52 values, about a day, with the histograms that it has seen before. If the histogram matches none of them, the score is 1 and the cell prints an alert. The model learns from each value after it scores it, so it needs no training data. It needs the range of the values: fewer than 40,000 passengers ride in 30 minutes.

The chart grows by one week at a time. The shaded areas are NAB's anomaly windows, which the model never sees.

In [ ]:
import matplotlib.pyplot as plt

try:
    from IPython.display import DisplayHandle
except ImportError:

    class DisplayHandle:
        def display(self, *a, **kw):
            pass

        def update(self, *a, **kw):
            pass


from pysad.models import RelativeEntropy
from pysad.utils import ArrayStreamer

model = RelativeEntropy(min_val=0, max_val=40_000)
week = 7 * 48  # values in one week
alerts = []

fig, ax = plt.subplots(figsize=(10, 3.5))
chart = DisplayHandle()
chart.display(fig)

for t, (x, label) in enumerate(ArrayStreamer(shuffle=False).iter(X, y)):
    if model.fit_score_partial(x) == 1.0:
        alerts.append(t)
        window = "in a NAB window" if label else "outside the windows"
        print(
            f"{times[t].astype(object):%a %d %b %Y %H:%M}  alert: {x[0]:,.0f} passengers, {window}"
        )

    if (t + 1) % week == 0 or t + 1 == len(X):
        seen = slice(0, t + 1)
        ax.clear()
        ax.fill_between(
            times[seen],
            0,
            1,
            where=y[seen] == 1,
            color="tab:orange",
            alpha=0.25,
            transform=ax.get_xaxis_transform(),
            label="NAB anomaly window",
        )
        ax.plot(times[seen], X[seen, 0], color="tab:blue", linewidth=0.6, label="passengers")
        ax.scatter(times[alerts], X[alerts, 0], color="tab:red", zorder=3, label="alert")
        ax.set_xlim(times[0], times[-1])
        ax.set_ylim(0, 40_000)
        ax.set_ylabel("passengers in 30 minutes")
        ax.legend(loc="upper left")
        chart.update(fig)

plt.close(fig)

## Compare the alerts with NAB's labels

In [ ]:
window_ids = np.cumsum(np.diff(y, prepend=0) == 1) * y  # 1 to 5 in the windows, 0 outside
caught = np.unique(window_ids[alerts])
print(f"{len(alerts)} alerts, {int(y[alerts].sum())} of them in NAB's anomaly windows")
print(f"{np.count_nonzero(caught)} of {window_ids.max()} windows have an alert")

The alerts outside the windows are not all false alarms. For example, NAB does not mark the 4th of July.

## Try it yourself

Edit a cell and run it again, for example:

- Change `max_val` or `window_size` of `RelativeEntropy`.
- Use another univariate detector, such as `StandardAbsoluteDeviation()` or `KNNCAD(probationary_period=750)`. Their scores are not 0 or 1, so compare each score with a threshold, or calibrate them as the [alerts guide](https://pysad.readthedocs.io/en/stable/guides/alerts.html) shows.

To run PySAD on your own data, install it with `pip install pysad`. The [documentation](https://pysad.readthedocs.io/en/stable/) lists all detectors, and the [repository](https://github.com/selimfirat/pysad) has more examples.